In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
df_train = pd.read_csv('/content/drive/MyDrive/DA Assignment Dataset/Train_final.csv')
df_test = pd.read_csv('/content/drive/MyDrive/DA Assignment Dataset/Test_final.csv')

In [3]:
import numpy as np
import pandas as pd

In [4]:
X_train = df_train.drop('Possibility', axis=1).values
y_train = df_train['Possibility'].values
X_test = df_test.drop('Possibility', axis=1).values
y_test = df_test['Possibility'].values
X_train

array([[-0.26956405, -0.14006516,  1.06304342, ..., -0.03241306,
         1.15828643,  0.60686342],
       [-1.18140582, -0.14006516,  1.06304342, ..., -0.03241306,
         1.15828643,  0.60686342],
       [ 0.03438321, -0.14006516, -1.69926488, ...,  1.50359145,
        -0.17093821,  0.60686342],
       ...,
       [-0.11759042, -0.14006516,  1.06304342, ...,  4.49880024,
         1.15828643,  0.60686342],
       [ 1.32615905, -0.14006516, -1.69926488, ..., -0.03241306,
        -1.34350728, -1.56829064],
       [ 0.49030409, -0.14006516, -0.12790078, ...,  0.04438716,
         0.30457761,  0.60686342]])

**Gaussian NB**

In [5]:
class GaussianNaiveBayes:
    def fit(self, X, y):
        """
        Fit the Gaussian Naive Bayes model to the training data.
        """
        self.classes = np.unique(y)
        self.means = {}
        self.variances = {}
        self.priors = {}

        for c in self.classes:
            X_c = X[y == c]
            self.means[c] = np.mean(X_c, axis=0)
            self.variances[c] = np.var(X_c, axis=0)
            self.priors[c] = X_c.shape[0] / float(X.shape[0])

    def _predict_log_proba(self, X):
        """
        Calculate the log probability of each class for each sample.
        """
        log_probs = []
        for c in self.classes:
            mean = self.means[c]
            variance = self.variances[c]
            prior = np.log(self.priors[c])
            # Compute the log probability of the data given the class
            log_prob = -0.5 * np.sum(np.log(2. * np.pi * variance))
            log_prob -= 0.5 * np.sum(((X - mean) ** 2) / variance, axis=1)
            log_probs.append(log_prob + prior)
        return np.array(log_probs).T

    def predict(self, X):
        """
        Predict class labels for the input data.
        """
        log_probs = self._predict_log_proba(X)
        return self.classes[np.argmax(log_probs, axis=1)]

    def predict_proba(self, X):
        """
        Predict class probabilities for the input data.
        """
        log_probs = self._predict_log_proba(X)
        # Convert log probabilities to probabilities
        exp_log_probs = np.exp(log_probs - np.max(log_probs, axis=1, keepdims=True))
        return exp_log_probs / np.sum(exp_log_probs, axis=1, keepdims=True)

model = GaussianNaiveBayes()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
probabilities = model.predict_proba(X_test)

**Bernoulli NB**

In [7]:
class BernoulliNaiveBayes:
    def __init__(self):
        self.class_prior_ = None
        self.feature_probs_ = None
        self.classes_ = None

    def fit(self, X, y):
        """
        Fit the Bernoulli Naive Bayes model according to the given training data.

        Parameters:
        X : numpy array of shape (n_samples, n_features)
        y : numpy array of shape (n_samples,)
        """
        # Get unique classes
        self.classes_, y_counts = np.unique(y, return_counts=True)
        self.class_prior_ = y_counts / len(y)

        # Initialize feature probabilities matrix
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        self.feature_probs_ = np.zeros((n_classes, n_features))

        # Calculate feature probabilities
        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.feature_probs_[i, :] = (X_c.sum(axis=0) + 1) / (X_c.shape[0] + 2)  # Add-one smoothing

    def predict(self, X):
        """
        Perform classification on samples in X.

        Parameters:
        X : numpy array of shape (n_samples, n_features)

        Returns:
        predictions : numpy array of shape (n_samples,)
        """
        # Calculate log probabilities
        log_class_prior = np.log(self.class_prior_)
        log_feature_probs = np.log(self.feature_probs_)
        log_feature_probs_neg = np.log(1 - self.feature_probs_)

        log_probs = np.dot(X, log_feature_probs.T) + np.dot(1 - X, log_feature_probs_neg.T)
        log_probs += log_class_prior

        # Return the class with the highest log probability
        return self.classes_[np.argmax(log_probs, axis=1)]

model = BernoulliNaiveBayes()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
data_frame = pd.DataFrame(y_pred)
data_frame.isna().sum()

<ipython-input-7-f7fdd09848d2>:41: RuntimeWarning: invalid value encountered in log
  log_feature_probs = np.log(self.feature_probs_)


,0
0,0


In [8]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

# f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print('Classification Report:')
print(report)

Accuracy: 0.7469865554010199
Classification Report:
              precision    recall  f1-score   support

         0.0       0.75      1.00      0.86      6445
         1.0       0.00      0.00      0.00      2183

    accuracy                           0.75      8628
   macro avg       0.37      0.50      0.43      8628
weighted avg       0.56      0.75      0.64      8628



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# SKLearn Naive Bayes Model


Gaussian


In [12]:
# compare to sklearn Naive Bayes Classifier
from sklearn.naive_bayes import GaussianNB
skmodel = GaussianNB()

In [13]:
skmodel.fit(X_train, y_train)
y_pred = skmodel.predict(X_test)

In [14]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print('Classification Report:')
print(report)

Accuracy: 0.78
Classification Report:
              precision    recall  f1-score   support

         0.0       0.79      0.97      0.87      6445
         1.0       0.72      0.24      0.36      2183

    accuracy                           0.78      8628
   macro avg       0.75      0.60      0.62      8628
weighted avg       0.77      0.78      0.74      8628



Bernoulli


In [9]:
from sklearn.naive_bayes import BernoulliNB
skmodel = BernoulliNB()

In [10]:
skmodel.fit(X_train, y_train)
y_pred = skmodel.predict(X_test)

In [11]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print('Classification Report:')
print(report)

Accuracy: 0.80
Classification Report:
              precision    recall  f1-score   support

         0.0       0.85      0.89      0.87      6445
         1.0       0.62      0.52      0.57      2183

    accuracy                           0.80      8628
   macro avg       0.73      0.71      0.72      8628
weighted avg       0.79      0.80      0.79      8628

